In [118]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Path to dataset files: /Users/joshua/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1


### Business Objective

Goal:
Predict which customers are likely to churn.

Business decision:
Prioritize retention actions toward high-risk customers.

Target variable: `churn`

### Problem Type

Task: Classification

Target: `churn`

Unit of observation: one row = one customer

In [119]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, precision_recall_curve
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

In [120]:
import sklearn.compose
dir(sklearn.compose)

['ColumnTransformer',
 'TransformedTargetRegressor',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_column_transformer',
 '_target',
 'make_column_selector',
 'make_column_transformer']

### Load Dataset

In [121]:
loaded = pd.read_csv(path + '/data.csv', index_col='customerID')
df = loaded.copy()
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
customerID,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Shallow Analysis

In [122]:
df.shape

(7043, 20)

In [123]:
df.info()

<class 'pandas.DataFrame'>
Index: 7043 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 

In [124]:
df.nunique().sort_values(ascending=False)

TotalCharges        6531
MonthlyCharges      1585
tenure                73
PaymentMethod          4
DeviceProtection       3
Contract               3
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
TechSupport            3
StreamingTV            3
StreamingMovies        3
PaperlessBilling       2
gender                 2
SeniorCitizen          2
PhoneService           2
Dependents             2
Partner                2
Churn                  2
dtype: int64

In [125]:
# TotalCharges is wrongly loaded as string.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df.TotalCharges.fillna(0)

In [126]:
# SeniorCitizen is loaded as numeric but casted as boolean for convenience.
df['SeniorCitizen'] = df['SeniorCitizen'].astype('bool')

In [127]:
# Change gender to Male for convenience.
df['Male'] = df['gender'] == 'Male'
df = df.drop('gender', axis=1)

In [128]:
# Cast all Yes/No features to boolean.
yes_no_cols = df.columns[df.isin(['Yes', 'No']).all()]
df[yes_no_cols] = df[yes_no_cols].replace({'Yes': True, 'No': False}).astype('bool')

In [129]:
str_features = df.select_dtypes('str')
str_category_cols = str_features.columns[str_features.nunique() > 2]
df[str_category_cols] = df[str_category_cols].astype('category')

In [130]:
df.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
tenure,7043.0,32.371149,24.559481,0.00,9.00,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.50,70.35,89.85,118.75
TotalCharges,7043.0,2279.734304,2266.794470,0.00,398.55,1394.55,3786.60,8684.80


In [131]:
df.describe(include=['bool', 'category']).T

,count,unique,top,freq
SeniorCitizen,7043,2,False,5901
Partner,7043,2,False,3641
Dependents,7043,2,False,4933
PhoneService,7043,2,True,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095
TechSupport,7043,3,No,3473


### Data Quality Check

In [132]:
df.isna().sum().sort_index(ascending=False)

tenure              0
TotalCharges        0
TechSupport         0
StreamingTV         0
StreamingMovies     0
SeniorCitizen       0
PhoneService        0
PaymentMethod       0
Partner             0
PaperlessBilling    0
OnlineSecurity      0
OnlineBackup        0
MultipleLines       0
MonthlyCharges      0
Male                0
InternetService     0
DeviceProtection    0
Dependents          0
Contract            0
Churn               0
dtype: int64

In [133]:
df.duplicated().sum()

np.int64(22)

In [134]:
df.index.is_unique

True

There are repeated values that come from different customers. Don't drop duplicates.

In [135]:
df['Churn'].value_counts()

Churn
False    5174
True     1869
Name: count, dtype: int64

In [136]:
df['Churn'].value_counts(normalize=True).round(2) * 100

Churn
False    73.0
True     27.0
Name: proportion, dtype: float64

Churn proportion is ~1/4. Use of tecniques to adjust imbalance could be implemented.

### Define X / y

In [137]:
X = df.copy().drop(columns='Churn')
y = df['Churn']

### EDA

In [138]:
pd.crosstab(df['Male'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
Male,,
False,73.0,27.0
True,74.0,26.0


Male and female genders both share a ~1/4 of churn proportion. 

In [139]:
pd.crosstab(df['SeniorCitizen'], df['Churn'], normalize='index').round(2) * 100


Churn,False,True
SeniorCitizen,,
False,76.0,24.0
True,58.0,42.0


~2/5 of senior citizens churned compared to the ~1/4 of non senior citizens who also did. Older people may churn more.

In [140]:
df.groupby('Churn')['tenure'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
False,5174.0,37.57,24.11,0.0,15.0,38.0,61.0,72.0
True,1869.0,17.98,19.53,1.0,2.0,10.0,29.0,72.0


Churning customers stayed ~18 months on average compared to the ~38 months of customers who didn't. May be related to the contract durations.

In [141]:
pd.crosstab(df['InternetService'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
InternetService,,
DSL,81.0,19.0
Fiber optic,58.0,42.0
No,93.0,7.0


~2/5 of customers who use fiber optic churned, compared to the ~1/5 of those who used DSL and the ~1/20 of those who didn't have internet service. Maybe the fiber optic service presented inconveniences.

In [142]:
pd.crosstab(df['OnlineSecurity'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
OnlineSecurity,,
No,58.0,42.0
No internet service,93.0,7.0
Yes,85.0,15.0


~2/5 of customers without online security churned compared to the 3/20 of those who had it and the ~1/20 who didn't have internet service. Online security may be of importance to customers who have internet service.

In [143]:
pd.crosstab(df['OnlineBackup'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
OnlineBackup,,
No,60.0,40.0
No internet service,93.0,7.0
Yes,78.0,22.0


2/5 of customers without an online backup churned compared to the ~1/5 of customers who had it and the ~1/20 who didn't have internet service. Online backups may be of importance to customers who have internet service.

In [144]:
pd.crosstab(df['DeviceProtection'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
DeviceProtection,,
No,61.0,39.0
No internet service,93.0,7.0
Yes,77.0,23.0


~2/5 of customers without device protection churned compared to the ~1/4 of customers who had it and the ~1/20 of those who didn't have internet service. Device protection may be of importance to customers who have internet service.

In [145]:
pd.crosstab(df['TechSupport'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
TechSupport,,
No,58.0,42.0
No internet service,93.0,7.0
Yes,85.0,15.0


~2/5 of customers without customer support churned compared to the ~3/20 who had it and the ~1/20 of those who didn't have internet service. Tech support may be of importance to customers who have internet service.

In [146]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index').round(2)

Churn,False,True
Contract,,
Month-to-month,0.57,0.43
One year,0.89,0.11
Two year,0.97,0.03


~9/20 of customers who were in a month-to-month contract churned compared to the ~1/10 who were in a one year plan and the ~1/20 who were in the two year plan. Contract duration may be of great importance to avoid customer churn.

In [147]:
pd.crosstab(df['PaperlessBilling'], df['Churn'], normalize='index').round(2)

Churn,False,True
PaperlessBilling,,
False,0.84,0.16
True,0.66,0.34


~1/3 of customers who received paperless billing churned compared to the ~3/20 of those who didn't. Worth investigating.

In [148]:
df.groupby('Churn')['MonthlyCharges'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
False,5174.0,61.27,31.09,18.25,25.10,64.43,88.4,118.75
True,1869.0,74.44,24.67,18.85,56.15,79.65,94.2,118.35


Customers who churned payed $74.60, $13.21 more than people who didn't. Could be related to the contract prices.

In [149]:
df.groupby('Churn')['TotalCharges'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
False,5174.0,2549.91,2329.95,0.00,572.9,1679.52,4262.85,8672.45
True,1869.0,1531.80,1890.82,18.85,134.5,703.55,2331.30,8684.80


Customers who churned pay $1531.8 of total charges on average, $1018.11 less than people who didn't. Could be related to the tenancy.

In [150]:
pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index').round(2) * 100

Churn,False,True
PaymentMethod,,
Bank transfer (automatic),83.0,17.0
Credit card (automatic),85.0,15.0
Electronic check,55.0,45.0
Mailed check,81.0,19.0


~9/20 of customers who payed via electronic check churned, a greater proportion than customers who payed with other methods.

In [151]:
df[df.select_dtypes('number').columns].corr().round(2)

,tenure,MonthlyCharges,TotalCharges
tenure,1.00,0.25,0.83
MonthlyCharges,0.25,1.00,0.65
TotalCharges,0.83,0.65,1.00


There is a high correlation between total charges and tenure.

### Train / test split

In [152]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [153]:
categorical_cols = X.select_dtypes('category').columns
numeric_cols = X.select_dtypes('number').columns

numeric_pipeline = Pipeline([('imputer', SimpleImputer()), ('scaler', StandardScaler())])
catergoric_pipeline = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, numeric_cols), ('categorical', catergoric_pipeline, categorical_cols)])

### Baseline

In [154]:
baseline = Pipeline([('preprocessing', preprocessor), ('dummy', DummyClassifier(strategy='most_frequent'))])
baseline.fit(X_train, y_train)
baseline.score(X_test, y_test)

0.7345635202271115

In [155]:
categorical_cols = X.select_dtypes('category').columns
numeric_cols = X.select_dtypes('number').columns
boolean_cols = X.select_dtypes('boolean').columns

numeric_pipeline = Pipeline([('imputer', SimpleImputer()), ('scaler', StandardScaler())])
catergoric_pipeline = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, numeric_cols), 
                                  ('categorical', catergoric_pipeline, categorical_cols),
                                  ('boolean', 'passthrough', boolean_cols)])

#### Simple model

In [191]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(random_state=42, fit_intercept=False, class_weight='balanced')),
])

param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='recall_weighted',
    n_jobs=-1
)

search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall_weighted'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fi

In [192]:
search.best_score_

np.float64(0.7523901459572169)

In [193]:
y_pred = search.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.90      0.73      0.80      1035
        True       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409



In [194]:
confusion_matrix(y_test, y_pred, labels=[True, False])

array([[294,  80],
       [284, 751]])

In [195]:
y_proba = search.predict_proba(X_test)[:, 1]

In [196]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_df = pd.DataFrame({'threshold': thresholds, 'tpr': tpr, 'fpr': fpr})
roc_df['threshold_group'] = roc_df['threshold'].round(1)
roc_df.groupby('threshold_group').agg(avg_tpr=('tpr', 'mean'), avg_fpr=('fpr', 'mean')).round(2)

,avg_tpr,avg_fpr
threshold_group,,
0.0,1.00,0.93
0.1,0.99,0.68
0.2,0.96,0.55
0.3,0.92,0.44
0.4,0.87,0.36
0.5,0.80,0.29
0.6,0.71,0.21
0.7,0.59,0.13
0.8,0.39,0.06


In [162]:
precision, recall, tresholds = precision_recall_curve(y_test, y_proba)
pr_df = pd.DataFrame({'threshold': tresholds, 'precision': precision[:-1], 'recall': recall[:-1]})
pr_df['threshold_group'] = pr_df['threshold'].round(1)
pr_df.groupby('threshold_group').agg(avg_precision=('precision', 'mean'), avg_recall=('recall', 'mean')).round(2).sort_values('threshold_group', ascending=False)


,avg_precision,avg_recall
threshold_group,,
0.9,0.76,0.12
0.8,0.71,0.38
0.7,0.62,0.60
0.6,0.55,0.71
0.5,0.50,0.79
0.4,0.46,0.87
0.3,0.43,0.93
0.2,0.39,0.96
0.1,0.34,0.99


Model comparison

In [176]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__class_weight': ['balanced', None]
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__class_weight': ['balanced', None], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the ver

In [177]:
y_pred = search.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.86      0.81      0.83      1035
        True       0.55      0.62      0.58       374

    accuracy                           0.76      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.77      0.76      0.77      1409



In [178]:
results = X_test.copy()

results['actual'] = y_test
results['predicted'] = y_pred
results['error'] = results['actual'] != results['predicted']

In [179]:
pd.crosstab(results['InternetService'], results['error'], normalize='index').round(2) * 100

error,False,True
InternetService,,
DSL,79.0,21.0
Fiber optic,67.0,33.0
No,89.0,11.0
